# Time-conditioned U-Net for DDPM

This notebook checks $\varepsilon_\theta(x_t, t)$, the denoiser from Ho et al., [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239) (2020), Appendix B.

The blocks and assembled `UNet` live in [`04_unet.py`](04_unet.py). Later notebooks import that file instead of redefining it.

It follows [`03_forward_noising.ipynb`](03_forward_noising.ipynb). We still **do not train**. The exit criterion is a paper-width U-Net that maps `[B, 3, 32, 32]` plus a timestep to a finite noise prediction of the same shape, with working gradients.


## 1. Setup


In [1]:
import importlib
import random
import sys
from dataclasses import dataclass

import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display

# Numbered .py files are not valid import identifiers.
sys.modules["model"] = importlib.import_module("04_unet")
from model import (
    AttentionBlock,
    Downsample,
    ResidualBlock,
    SinusoidalTimeEmbedding,
    UNet,
    Upsample,
    group_norm,
)


@dataclass(frozen=True)
class Config:
    seed: int = 42
    image_size: int = 32
    in_channels: int = 3
    num_steps: int = 1000
    # Paper-width CIFAR-10 U-Net (Tier B). Changing base_channels to 32 is a debug switch, not a new model.
    base_channels: int = 128
    channel_mults: tuple[int, ...] = (1, 2, 2, 2)
    num_res_blocks: int = 2
    attention_resolutions: tuple[int, ...] = (16,)
    dropout: float = 0.1


cfg = Config()

# Same seed everywhere so later random tensors stay reproducible.
random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)

# A full paper-width backward pass is cheap on GPU; keep it smaller on CPU.
test_batch_size = 4 if device.type == "cuda" else 2

print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Test batch size: {test_batch_size}")
print(cfg)


PyTorch: 2.13.0
Device: cpu
CUDA available: False
Test batch size: 2
Config(seed=42, image_size=32, in_channels=3, num_steps=1000, base_channels=128, channel_mults=(1, 2, 2, 2), num_res_blocks=2, attention_resolutions=(16,), dropout=0.1)


## 2. GroupNorm helper

Diffusion training uses GroupNorm instead of BatchNorm so normalization does not depend on the current batch. The number of groups must divide the channel count. `group_norm` is imported from [`04_unet.py`](04_unet.py).


In [2]:
norm = group_norm(256)
assert isinstance(norm, nn.GroupNorm)
assert 256 % norm.num_groups == 0
print(f"256 channels -> {norm.num_groups} groups")


256 channels -> 32 groups


## 3. Sinusoidal timestep embedding

The U-Net is conditioned on $t$ through a sinusoidal embedding followed by a two-layer MLP. The class lives in [`04_unet.py`](04_unet.py); here we only check shapes.


In [3]:
time_dim = cfg.base_channels * 4
time_mlp = nn.Sequential(
    SinusoidalTimeEmbedding(cfg.base_channels),
    nn.Linear(cfg.base_channels, time_dim),
    nn.SiLU(),
    nn.Linear(time_dim, time_dim),
).to(device)

for batch_size in (1, test_batch_size):
    timesteps = torch.randint(0, cfg.num_steps, (batch_size,), device=device)
    embedding = time_mlp(timesteps)
    print(f"batch {batch_size}: timesteps {tuple(timesteps.shape)} -> embedding {tuple(embedding.shape)}")
    assert embedding.shape == (batch_size, time_dim)
    assert torch.isfinite(embedding).all()


batch 1: timesteps (1,) -> embedding (1, 512)
batch 2: timesteps (2,) -> embedding (2, 512)


## 4. Residual block

Time is added after the first convolution. The last convolution is zero-initialized so the block starts near an identity. Implementation: [`04_unet.py`](04_unet.py).


In [4]:
res_same = ResidualBlock(128, 128, time_dim, cfg.dropout).to(device)
res_change = ResidualBlock(128, 256, time_dim, cfg.dropout).to(device)
features = torch.randn(test_batch_size, 128, cfg.image_size, cfg.image_size, device=device)
time_emb = time_mlp(torch.randint(0, cfg.num_steps, (test_batch_size,), device=device))

out_same = res_same(features, time_emb)
out_change = res_change(features, time_emb)
print(f"same channels:    {tuple(features.shape)} -> {tuple(out_same.shape)}")
print(f"changed channels: {tuple(features.shape)} -> {tuple(out_change.shape)}")
assert out_same.shape == features.shape
assert out_change.shape == (test_batch_size, 256, cfg.image_size, cfg.image_size)
assert torch.isfinite(out_same).all() and torch.isfinite(out_change).all()


same channels:    (2, 128, 32, 32) -> (2, 128, 32, 32)
changed channels: (2, 128, 32, 32) -> (2, 256, 32, 32)


## 5. Self-attention at $16 \times 16$

Attention runs at $16 \times 16$ (and in the $4 \times 4$ middle block). Implementation: [`04_unet.py`](04_unet.py).


In [5]:
attn = AttentionBlock(256).to(device)
attn_in = torch.randn(test_batch_size, 256, 16, 16, device=device)
attn_out = attn(attn_in)
print(f"attention: {tuple(attn_in.shape)} -> {tuple(attn_out.shape)}")
assert attn_out.shape == attn_in.shape
assert torch.isfinite(attn_out).all()


attention: (2, 256, 16, 16) -> (2, 256, 16, 16)


## 6. Downsample and upsample

Stride-2 $3 \times 3$ down; nearest-neighbor $\times 2$ plus $3 \times 3$ up. Implementation: [`04_unet.py`](04_unet.py).


In [6]:
down = Downsample(128).to(device)
up = Upsample(128).to(device)
down_in = torch.randn(test_batch_size, 128, 32, 32, device=device)
down_out = down(down_in)
up_out = up(down_out)
print(f"down: {tuple(down_in.shape)} -> {tuple(down_out.shape)}")
print(f"up:   {tuple(down_out.shape)} -> {tuple(up_out.shape)}")
assert down_out.shape == (test_batch_size, 128, 16, 16)
assert up_out.shape == down_in.shape


down: (2, 128, 32, 32) -> (2, 128, 16, 16)
up:   (2, 128, 16, 16) -> (2, 128, 32, 32)


## 7. Assemble the U-Net

Recommended tensor path:

| Stage | Resolution | Channels | Operations |
| --- | --- | --- | --- |
| Input convolution | 32×32 | 128 | `3 → 128` |
| Encoder level 1 | 32×32 | 128 | 2 residual blocks, save skips, downsample |
| Encoder level 2 | 16×16 | 256 | 2 residual blocks + attention, save skips, downsample |
| Encoder level 3 | 8×8 | 256 | 2 residual blocks, save skips, downsample |
| Encoder level 4 | 4×4 | 256 | 2 residual blocks, no further downsample |
| Middle | 4×4 | 256 | ResBlock → attention → ResBlock |
| Decoder | 4→8→16→32 | mirrored | concatenate skips, 3 residual blocks per level, upsample |
| Output | 32×32 | 3 | GroupNorm → SiLU → zero-initialized 3×3 conv |

With 2 encoder residual blocks per level, the decoder uses 3 residual blocks per level so every stored skip is consumed.

The encoder / middle / decoder assembly lives in [`04_unet.py`](04_unet.py). This cell only instantiates it.


In [7]:
model = UNet(cfg).to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Parameters: {parameter_count:,} ({parameter_count / 1e6:.2f} M)")


Parameters: 35,746,307 (35.75 M)


## 8. Shape, gradient, and parameter checks

These are the report's required tests before any training loop. The output convolution is zero-initialized, so the first forward pass should stay near zero. That is expected, not a bug.


In [8]:
images = torch.randn(test_batch_size, cfg.in_channels, cfg.image_size, cfg.image_size, device=device)
timesteps = torch.randint(0, cfg.num_steps, (test_batch_size,), device=device)
predicted_noise = model(images, timesteps)

print(f"input:  {tuple(images.shape)}")
print(f"output: {tuple(predicted_noise.shape)}")
print(f"output mean/std: {predicted_noise.mean().item():.4f} / {predicted_noise.std().item():.4f}")

assert predicted_noise.shape == images.shape
assert torch.isfinite(predicted_noise).all()

predicted_noise.mean().backward()
assert any(parameter.grad is not None for parameter in model.parameters())
print("Backward pass produced gradients.")

# Paper CIFAR-10 width is about 35.7 million parameters.
assert 30e6 < parameter_count < 42e6

trace = pd.DataFrame(
    [
        {"stage": "input convolution", "resolution": "32x32", "channels": 128},
        {"stage": "encoder level 1", "resolution": "32x32 -> 16x16", "channels": 128},
        {"stage": "encoder level 2 + attention", "resolution": "16x16 -> 8x8", "channels": 256},
        {"stage": "encoder level 3", "resolution": "8x8 -> 4x4", "channels": 256},
        {"stage": "encoder level 4 + middle", "resolution": "4x4", "channels": 256},
        {"stage": "decoder", "resolution": "4 -> 8 -> 16 -> 32", "channels": "mirrored"},
        {"stage": "output convolution", "resolution": "32x32", "channels": 3},
    ]
)
display(trace)


input:  (2, 3, 32, 32)
output: (2, 3, 32, 32)
output mean/std: 0.0000 / 0.0000


Backward pass produced gradients.


,stage,resolution,channels
0,input convolution,32x32,128
1,encoder level 1,32x32 -> 16x16,128
2,encoder level 2 + attention,16x16 -> 8x8,256
3,encoder level 3,8x8 -> 4x4,256
4,encoder level 4 + middle,4x4,256
5,decoder,4 -> 8 -> 16 -> 32,mirrored
6,output convolution,32x32,3


## 9. What this notebook proved

- [`04_unet.py`](04_unet.py) is the source of the paper-width U-Net.
- Timestep embeddings have shape `[B, 4 * base_channels]` for batch sizes 1 and greater.
- Residual blocks accept a time vector, change channel counts when asked, and stay finite.
- Attention preserves a `16x16` feature map.
- Down/up sampling change spatial size by 2 and back.
- The assembled paper-width U-Net has about 35.7M parameters and maps `[B, 3, 32, 32]` plus $t$ to a finite tensor of the same shape.
- A backward pass produces gradients.

Next: [`06_train.ipynb`](06_train.ipynb) — one-batch overfit first, then AMP, EMA, checkpoints, and the production loop. Do not start the long GPU run until that overfit loss drops.
